In [23]:
import pandas as pd
from datetime import timedelta, date
from difflib import SequenceMatcher
import math
from typing import Dict, Tuple, List, Any, Iterable, Optional, Union


df = pd.read_csv("in/tables/suggestion_funnel.csv",
                 usecols=["eventDate", "division", "searchString", "suggestionView", "searchCount", "suggestionClicks", "dealClicks", "dealPurchased" ])

### Read all data from table, it is not necaserry, because we will need -7 days only

In [101]:
df.head()

,eventDate,division,searchString,suggestionView,searchCount,suggestionClicks,dealClicks,dealPurchased
0,2025-08-06,seattle,lulule,lulul,1,0,0,0
1,2025-08-06,seattle,shark stainstriker portable carpet upholstery ...,window cleaning,1,0,0,0
2,2025-08-06,seattle,tukw,car tune up,1,0,0,0
3,2025-08-06,seattle,willows golf,golf lessons,1,0,0,0
4,2025-08-06,stlouis,ji,jiffy lube,1,0,0,0


### Try to search in data

In [19]:
df.query("eventDate == '2025-08-08' and division == 'chicago' and searchString == 'massage'").sort_values("searchCount", ascending = False)

,eventDate,division,searchString,suggestionView,searchCount,suggestionClicks,dealClicks,dealPurchased
1096491,2025-08-08,chicago,massage,couples massage,125,9,6,0
3698041,2025-08-08,chicago,massage,deep tissue massage,117,4,3,1
529596,2025-08-08,chicago,massage,lymphatic massage,116,3,2,0
356228,2025-08-08,chicago,massage,massage,113,15,23,1
5045302,2025-08-08,chicago,massage,foot massage,110,0,0,0
1550394,2025-08-08,chicago,massage,thai massage,8,0,0,0
5212263,2025-08-08,chicago,massage,massage and facial,7,0,0,0
4725306,2025-08-08,chicago,massage,manicure,7,0,0,0
2351398,2025-08-08,chicago,massage,mani pedi,6,0,0,0
1229869,2025-08-08,chicago,massage,mall of america,5,0,0,0


# `build_index_over_range` — Weekly Index Aggregation

**TL;DR:** Compress many daily rows into **one row per key**  
`key = (division, searchString, suggestionView)`  
over a chosen date range `[start_date, end_date]`.  
Take **`end_date` as the base**, then fold older days in:
- if the day **exists** → **average** with the current aggregate
- if the day is **missing** → **decay** the current aggregate by γ (`decay_if_missing`, 0<γ<1)

---

## Why / When
Use this to produce a **stable weekly index** that reflects the latest performance while still remembering recent history. Missing days aren’t treated as zeros; they’re **down-weighted**.

---

## Inputs
- **`df`**: table with at least  
  `eventDate, division, searchString, suggestionView` + numeric **metrics**  
  (e.g., `searchCount, suggestionClicks, dealClicks, dealPurchased`).
- **`start_date`**, **`end_date`**: inclusive range (e.g., `2025-08-01` to `2025-08-08`).
- **`metrics`**: which numeric columns to aggregate (default: above).
- **`decay_if_missing`** (`γ`): penalty for missing days (e.g., `0.85`).

---

## Output
DataFrame with **one row per key**:  
`division, searchString, suggestionView, indexAsOf=end_date, <aggregated metrics>`

---

## Exact Rules per Key
1. **Base** = metric values from **`end_date`** (key must exist on `end_date`, otherwise it’s dropped).
2. For each earlier day `d = end_date-1, …, start_date`:
   - If `(key, d)` **exists** → `agg = (agg + values[d]) / 2`
   - If **missing** → `agg = agg * γ`

> Example (single metric): base=100  
> day−1 exists: 60 → `(100+60)/2 = 80`  
> day−2 missing: `80 * 0.85 = 68`

---

## Daily Duplicates
If multiple raw rows exist for the same `(key, day)`, they are **averaged** first (`.mean()`), then the rolling logic above is applied.

---

## Edge Cases
- No data in range → returns an **empty** DataFrame with expected columns.  
- No keys present on `end_date` → empty (nothing to anchor to).

---

## Complexity
Roughly **O(K × D)** where `K` = number of keys seen on `end_date`, `D` = days in the range.

---

## Tuning
- **`decay_if_missing` (γ)**: smaller → harsher penalty for gaps; larger (e.g., 0.95) → gentler.  
- Swap daily `.mean()` to `.sum()` if you want day totals to accumulate before rolling.

---

In [103]:
def build_index_over_range(
    df: pd.DataFrame,
    start_date: str | date,
    end_date: str | date,
    metrics=("searchCount", "suggestionClicks", "dealClicks", "dealPurchased"),
    key_cols=("division", "searchString", "suggestionView"),
    date_col="eventDate",
    decay_if_missing=0.85,   # γ < 1 for days where the key is missing
):
    """
    Build a 1-row-per-key index over a date range [start_date, end_date].

    Rules:
      - Key = (division, searchString, suggestionView)
      - Base = values from end_date (must exist)
      - For each older day in the range:
          * If the (key, day) exists: average with the current aggregate
          * If missing: multiply the current aggregate by decay_if_missing (γ)

    Daily duplicates for the same key are aggregated using .mean() before rolling logic.
    """

    # 1) Normalize dates and filter to the given inclusive range
    work = df.copy()
    work[date_col] = pd.to_datetime(work[date_col]).dt.date
    start_date = pd.to_datetime(start_date).date()
    end_date = pd.to_datetime(end_date).date()

    work = work[(work[date_col] >= start_date) & (work[date_col] <= end_date)]
    if work.empty:
        return pd.DataFrame(columns=list(key_cols) + ["indexAsOf"] + list(metrics))

    # 2) If multiple rows per (key, day) exist → aggregate them by mean
    grouped_daily = (
        work.groupby(list(key_cols) + [date_col], as_index=False)[list(metrics)].mean()
    )

    # 3) Keep only keys that exist on the end_date (these form the base)
    mask_last = grouped_daily[date_col] == end_date
    if not mask_last.any():
        # No base rows on end_date → nothing to build
        return pd.DataFrame(columns=list(key_cols) + ["indexAsOf"] + list(metrics))

    keys_last_day = grouped_daily.loc[mask_last, list(key_cols)].drop_duplicates()
    grouped_daily = grouped_daily.merge(keys_last_day, on=list(key_cols), how="inner")

    # 4) Prepare a MultiIndex: (division, searchString, suggestionView, eventDate)
    idx_cols = list(key_cols) + [date_col]
    grouped_daily = grouped_daily.set_index(idx_cols).sort_index()

    # 5) Rolling aggregation: start from end_date, then step back day by day to start_date
    all_days = pd.date_range(start=start_date, end=end_date, freq="D").date
    results = []

    for k_vals in keys_last_day.itertuples(index=False, name=None):
        k = tuple(k_vals)

        # Base values from end_date (must exist by construction)
        base = grouped_daily.loc[(k + (end_date,)), list(metrics)].astype(float).copy()
        agg_vals = base.copy()

        # Iterate older days from end_date-1 down to start_date
        for d in reversed(all_days):
            if d == end_date:
                continue
            key_day = k + (d,)
            if key_day in grouped_daily.index:
                # Existing day → average with current aggregate
                daily_vals = grouped_daily.loc[key_day, list(metrics)].astype(float)
                agg_vals = (agg_vals + daily_vals) / 2.0
            else:
                # Missing day → apply decay γ
                agg_vals *= decay_if_missing

        out_row = dict(zip(key_cols, k))
        out_row.update(agg_vals.to_dict())
        out_row["indexAsOf"] = end_date
        results.append(out_row)

    index_df = pd.DataFrame(results)[list(key_cols) + ["indexAsOf"] + list(metrics)]
    return index_df

## Be carefull, it takes time !!!

In [25]:
index_df = build_index_over_range(
     df,
     start_date="2025-08-01",
     end_date="2025-08-08",
     decay_if_missing=0.85
)
display(index_df.head())

,division,searchString,suggestionView,indexAsOf,searchCount,suggestionClicks,dealClicks,dealPurchased
0,Arkansas City,kids,kids,2025-08-08,0.320577,0.0,0.0,0.0
1,Arkansas City,kids magazines,kids magazines,2025-08-08,0.320577,0.0,0.0,0.0
2,Inland Empire,facial,acne facial,2025-08-08,0.320577,0.0,0.0,0.0
3,Inland Empire,facial,back facial,2025-08-08,0.320577,0.0,0.0,0.0
4,Inland Empire,facial,facial,2025-08-08,0.320577,0.0,0.0,0.0


### try to search in index

In [73]:
index_df.query("division == 'chicago' and searchString == 'mass'").sort_values("searchCount", ascending = False)

,division,searchString,suggestionView,indexAsOf,searchCount,suggestionClicks,dealClicks,dealPurchased
74017,chicago,mass,massage,2025-08-08,21.382812,9.054688,10.960938,1.6875
74006,chicago,mass,couples massage,2025-08-08,15.828125,1.734375,1.523438,0.1250
74010,chicago,mass,lymphatic massage,2025-08-08,15.718750,0.367188,0.343750,0.0000
74008,chicago,mass,foot massage,2025-08-08,14.507812,0.007812,0.015625,0.0000
74015,chicago,mass,manicure,2025-08-08,5.976562,0.000000,0.000000,0.0000
74026,chicago,mass,movies,2025-08-08,4.890625,0.000000,0.000000,0.0000
74030,chicago,mass,museum,2025-08-08,4.640625,0.000000,0.000000,0.0000
74014,chicago,mass,mani pedi,2025-08-08,3.617188,0.000000,0.000000,0.0000
74011,chicago,mass,main event,2025-08-08,3.601562,0.000000,0.000000,0.0000
74007,chicago,mass,deep tissue massage,2025-08-08,3.382812,0.320312,0.281250,0.0000


### remapping to dict for quick access

In [62]:
# Build a flat dict: (division, searchString, suggestionView) -> row dict of metrics (+ indexAsOf)
KEY_COLS = ["division", "searchString", "suggestionView"]

def build_index_map(index_df, drop_index_as_of=False):
    """
    Create a fast lookup map from index_df.
    Key: (division, searchString, suggestionView)
    Value: dict of the remaining columns (metrics, optionally indexAsOf).
    """
    df = index_df.copy()
    if drop_index_as_of and "indexAsOf" in df.columns:
        df = df.drop(columns=["indexAsOf"])
    # Ensure unique keys; if not, keep the last occurrence
    df = df.drop_duplicates(subset=KEY_COLS, keep="last")
    # MultiIndex -> dict where keys are tuples
    return (
        df.set_index(KEY_COLS)
          .sort_index()
          .to_dict(orient="index")  # {(div, search, sugg): {col: value, ...}, ...}
    )

In [63]:
index_map = build_index_map(index_df, drop_index_as_of=False)

In [65]:
# test
row = index_map[("chicago", "mass", "couples massage")]
display(row)

{'indexAsOf': datetime.date(2025, 8, 8),
 'searchCount': 15.828125,
 'suggestionClicks': 1.734375,
 'dealClicks': 1.5234375,
 'dealPurchased': 0.125}

# Pure-Python helpers for lookup, sorting, and feature engineering

**TL;DR:** This module lets you:
- **Filter** records from a prebuilt `index_map` (no pandas needed),
- **Sort** them by a chosen numeric field,
- Compute simple **text features** (similarity, overlap, exact/prefix checks),
- Compute **metric features** (log-scaled counts, rates),
- **Enrich** keys/rows with these features in one call.

---

## Data Structures

- **`Key = (division, searchString, suggestionView)`**  
  A unique identifier for one suggestion candidate within a division.

- **`Row = Dict[str, Any]`**  
  A record holding metrics (e.g., `searchCount`, `suggestionClicks`, …) and optional metadata (e.g., `indexAsOf`).

- **`index_map: Dict[Key, Row]`**  
  A fast lookup table:  
  `("chicago", "massage", "couples massage") → {"searchCount": 113, "suggestionClicks": 15, ...}`

---

## Lookup & Filtering (no pandas)

### `iter_rows_from_map(index_map, division=None, searchString=None) -> Iterable[(Key, Row)]`
Yield (key, value) pairs filtered by optional `division` and/or `searchString`.

- If `division` is given, only rows with that division pass.
- If `searchString` is given, only rows with that search string pass.
- Useful as a building block for downstream sorting or ranking.

### `sort_rows(rows, sort_by="searchCount", reverse=True) -> List[Row]`
Sort an iterable of `(Key, Row)` pairs by a numeric field in the **value dict**.

- Default sort: descending by `searchCount`.
- Merges key fields back into each output row:
  `{ "division": ..., "searchString": ..., "suggestionView": ..., **metrics }`.
- Returns a **list of rows** (no tuples, ready for feature engineering).

---

## Text Feature Helpers

All functions are safe on arbitrary strings (they lower-case and trim internally).

- **`char_similarity(a, b) -> float`**  
  Character-level similarity in **[0, 1]** using `difflib.SequenceMatcher`.

- **`word_overlap_count(a, b) -> int`**  
  Count of overlapping **unique** words (set intersection of tokens).

- **`starts_with_same_word(a, b) -> int`**  
  `1` if both strings start with the **same first word**, else `0`.

- **`is_exact_match(a, b) -> int`**  
  `1` if strings are identical after **lowercasing + trimming**, else `0`.

> These features are lightweight and deterministic; ideal for first-pass scoring.

---

## Metric Features

### `compute_metric_features(val: Row) -> Row`
Derive rate-like features from aggregated counters in `val`.

- Reads (safely defaults to 0):
  - `searchCount`, `suggestionClicks`, `dealClicks`, `dealPurchased`
- Emits:
  - `log_searchCount` — `log1p(searchCount)` (stabilizes heavy tails)
  - `click_rate` — `suggestionClicks / max(searchCount, 1)`
  - `deal_click_rate` — `dealClicks / max(searchCount, 1)`
  - `purchase_rate` — `dealPurchased / max(searchCount, 1)`
  - `has_activity` — `1` if any clicks/purchases > 0, else `0`

> These features are robust to zeros and missing fields.

---

## Feature Enrichment

### `add_features(items, index_map) -> List[Row]`
Enrich **keys or rows** with both text & metric features.

- **Inputs:**
  - `items`: either
    - a list of `Key` tuples `(division, searchString, suggestionView)`, **or**
    - a list of row dicts that already include those three fields.
  - `index_map`: lookup for metrics by key.

- **Behavior:**
  1. For each item:
     - If it is a `Key` → pull metrics from `index_map[key]` (or `{}` if missing).
     - If it is a dict → merge it with `index_map[key]`; the pulled payload overrides same-name fields.
  2. Compute **text features** using `searchString` and `suggestionView`:
     - `text_similarity`, `length_diff`, `word_overlap_count`, `starts_with_same_word`, `is_exact_match`
     - `missing_in_index` flag = `1` if the key wasn’t found in `index_map`.
  3. Compute **metric features** via `compute_metric_features` (safe even if metrics were missing → treated as zeros).
  4. Return a list of **fully enriched rows**.

- **Output row contains:**
  - Key fields: `division`, `searchString`, `suggestionView`
  - Original metrics from `index_map` (if present)
  - Text features (above)
  - Metric features (above)
  - `missing_in_index` flag

---

## Edge Cases
- **Missing key in `index_map`** → metrics default to 0 and `missing_in_index=1`.
- **Malformed item** (not a 3-tuple or dict) → raises a `ValueError`.
- **Empty input** → returns an empty list.

---

## Complexity
- `iter_rows_from_map`: **O(N)** scan over `index_map`.
- `sort_rows`: **O(M log M)** for `M` filtered items.
- `add_features`: **O(M)** with small constant factors (string ops + a few logs/divisions).

---

## Example Usage

```python
# Filter & sort (no pandas)
rows = sort_rows(
    iter_rows_from_map(index_map, division="chicago", searchString="massage"),
    sort_by="searchCount",
    reverse=True,
)

# Enrich with features
featured_rows = add_features(rows, index_map)

# Now you can pass to your ranker:
# ranked = ranker(featured_rows)

In [105]:
# Types for clarity
Key = Tuple[str, str, str]   # (division, searchString, suggestionView)
Row = Dict[str, Any]         # row payload: metrics + optional indexAsOf, etc.

# ---------- Lookup & filtering (no pandas) ----------
def iter_rows_from_map(index_map: Dict[Key, Row],
                       division: Optional[str] = None,
                       searchString: Optional[str] = None) -> Iterable[Tuple[Key, Row]]:
    """Yield (key, value) pairs filtered by optional division and searchString."""
    for (div, s, v), val in index_map.items():
        if division is not None and div != division:
            continue
        if searchString is not None and s != searchString:
            continue
        yield (div, s, v), val

def sort_rows(rows: Iterable[Tuple[Key, Row]],
              sort_by: str = "searchCount",
              reverse: bool = True) -> List[Row]:
    """Sort rows by a numeric field in value dict (default: searchCount)."""
    items = sorted(rows, key=lambda kv: kv[1].get(sort_by, 0), reverse=reverse)
    out: List[Row] = []
    for (div, s, v), val in items:
        rec = {"division": div, "searchString": s, "suggestionView": v}
        rec.update(val)  # merge metrics (searchCount, suggestionClicks, ...)
        out.append(rec)
    return out

# ---------- Text feature helpers ----------
def char_similarity(a: str, b: str) -> float:
    """Character-level similarity using SequenceMatcher ratio [0..1]."""
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

def word_overlap_count(a: str, b: str) -> int:
    """Count of overlapping unique words."""
    set1 = set(str(a).lower().split())
    set2 = set(str(b).lower().split())
    return len(set1 & set2)

def starts_with_same_word(a: str, b: str) -> int:
    """1 if both strings start with the same first word, else 0."""
    w1 = str(a).strip().split()
    w2 = str(b).strip().split()
    return int(len(w1) > 0 and len(w2) > 0 and w1[0].lower() == w2[0].lower())

def is_exact_match(a: str, b: str) -> int:
    """1 if lowercase-trimmed strings are identical, else 0."""
    return int(str(a).strip().lower() == str(b).strip().lower())

# ---------- Metric features ---------
def compute_metric_features(val: Row) -> Row:
    """Derive rate-like features from aggregated counters."""
    sc = float(val.get("searchCount", 0) or 0.0)
    suclk = float(val.get("suggestionClicks", 0) or 0.0)
    dclk = float(val.get("dealClicks", 0) or 0.0)
    purch = float(val.get("dealPurchased", 0) or 0.0)
    denom = sc if sc > 0 else 1.0

    return {
        "log_searchCount": math.log1p(sc),
        "click_rate": suclk / denom,
        "deal_click_rate": dclk / denom,
        "purchase_rate": purch / denom,
        "has_activity": int((suclk + dclk + purch) > 0),
    }


def add_features(items: List[Union[Key, Row]], index_map: Dict[Key, Row]) -> List[Row]:
    """
    Enrich keys/rows with text & metric features.
    - items: list of (division, searchString, suggestionView) tuples OR list of row dicts
    - index_map: {(division, searchString, suggestionView) -> metrics ...}
    """
    enriched: List[Row] = []
    for it in items:
        # Build base row + pull metrics from index_map
        if isinstance(it, tuple) and len(it) == 3:
            div, s, v = it
            payload = index_map.get(it, {})
            missing = int(not payload)
            base = {"division": div, "searchString": s, "suggestionView": v, **payload}
        elif isinstance(it, dict):
            div, s, v = it.get("division"), it.get("searchString"), it.get("suggestionView")
            key = (div, s, v)
            payload = index_map.get(key, {})
            missing = int(not payload)
            base = {**it, **payload}  # payload overrides if present
        else:
            raise ValueError("Each item must be a (division, searchString, suggestionView) tuple or a row dict.")

        # Text features
        s_txt = base.get("searchString", "")
        v_txt = base.get("suggestionView", "")
        feats = {
            "text_similarity": char_similarity(s_txt, v_txt),
            "length_diff": abs(len(str(s_txt)) - len(str(v_txt))),
            "word_overlap_count": word_overlap_count(s_txt, v_txt),
            "starts_with_same_word": starts_with_same_word(s_txt, v_txt),
            "is_exact_match": is_exact_match(s_txt, v_txt),
            "missing_in_index": missing,
        }

        # Metric-derived features (safe even if counts are missing → treated as 0)
        feats.update(compute_metric_features(base))

        enriched.append({**base, **feats})
    return enriched

# Ranker

In [97]:
DEFAULT_IMPORTANCE_WEIGHTS: Dict[str, float] = {
    "starts_with_same_word": 0.52,
    "suggestionClicks":      0.33,
    "text_similarity":       0.125,
    "searchCount":           0.10,
    "dealClicks":            0.3,
    "word_overlap_count":    0.05,
    "length_diff":           0.02,
}

# Count-like features we lightly stabilize with log1p
_COUNT_FEATURES = {"suggestionClicks", "searchCount", "dealClicks"}

def ranker(
    featured_rows: List[Dict[str, Any]],
    weights: Dict[str, float] = DEFAULT_IMPORTANCE_WEIGHTS,
    top_k: Optional[int] = None,
    attach_score: bool = True,
) -> List[Dict[str, Any]]:
    """
    Rank rows by a simple weighted sum using the provided feature importances.
    Only features listed in `weights` are used.

    score = Σ_f ( weight[f] * value_f_transformed )

    - Counts (suggestionClicks, searchCount, dealClicks): value := log1p(value)
    - length_diff: value := -length_diff (smaller diff is better)
    - text/binary features are used as-is.
    """
    scored = []
    for r in featured_rows:
        s = 0.0
        for f, w in weights.items():
            v = float(r.get(f, 0.0))
            if f in _COUNT_FEATURES:
                v = math.log1p(v)
            elif f == "length_diff":
                v = -v
            s += w * v
        scored.append((s, r))

    scored.sort(key=lambda t: t[0], reverse=True)
    out = []
    for s, r in scored[: (top_k or len(scored))]:
        out.append({**r, "score": s} if attach_score else r)
    return out

# Retrieval data

In [106]:
results = [
    ("chicago", "mass", "couples massage"),
    ("chicago", "mass", "massage"),
    ("chicago", "mass", "lymphatic massage"),
    ("chicago", "mass", "foot massage"),
    ("chicago", "mass", "masserati"),  # missing in index
]

featured_rows = add_features(results, index_map)

In [107]:
featured_rows

[{'division': 'chicago',
  'searchString': 'mass',
  'suggestionView': 'couples massage',
  'indexAsOf': datetime.date(2025, 8, 8),
  'searchCount': 15.828125,
  'suggestionClicks': 1.734375,
  'dealClicks': 1.5234375,
  'dealPurchased': 0.125,
  'text_similarity': 0.42105263157894735,
  'length_diff': 11,
  'word_overlap_count': 0,
  'starts_with_same_word': 0,
  'is_exact_match': 0,
  'missing_in_index': 0,
  'log_searchCount': 2.8230515937967167,
  'click_rate': 0.10957551826258638,
  'deal_click_rate': 0.096248766041461,
  'purchase_rate': 0.007897334649555774,
  'has_activity': 1},
 {'division': 'chicago',
  'searchString': 'mass',
  'suggestionView': 'massage',
  'indexAsOf': datetime.date(2025, 8, 8),
  'searchCount': 21.3828125,
  'suggestionClicks': 9.0546875,
  'dealClicks': 10.9609375,
  'dealPurchased': 1.6875,
  'text_similarity': 0.7272727272727273,
  'length_diff': 3,
  'word_overlap_count': 0,
  'starts_with_same_word': 0,
  'is_exact_match': 0,
  'missing_in_index': 0,

# Ranked result

In [100]:
# Example:
ranked = ranker(featured_rows)
for r in ranked:
    print(r["suggestionView"], "->", r["score"])

massage -> 1.8478851183666813
couples massage -> 0.7245713100040199
lymphatic massage -> 0.26112078019811247
foot massage -> 0.18385375522683503
masserati -> -0.023076923076923078
